In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from pyvi import ViTokenizer

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

data = []
with open("train_data.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        item["question"] = ViTokenizer.tokenize(item["question"])
        data.append(item)

label_mapping = {"Trẻ em": 0, "Bình đẳng giới": 1, "Dân số": 2, "Hôn nhân và gia đình": 3, "Phòng, chống bạo lực gia đình": 4, "Khác": 5}
for item in data:
    item["label"] = label_mapping[item["label"]]

def preprocess_function(examples):
    return tokenizer(examples["question"], truncation=True, padding="max_length", max_length=128)


In [3]:
len(data)

4357

In [ ]:
from sklearn.model_selection import train_test_split

question = [item["question"] for item in data]
labels = [item["label"] for item in data]

train_texts, val_test_texts, train_labels, val_test_labels = train_test_split(
    question, labels, test_size=0.3, stratify=labels, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    val_test_texts, val_test_labels, test_size=0.5, stratify=val_test_labels, random_state=42
)

train_dataset = Dataset.from_dict({"question": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"question": val_texts, "label": val_labels})
test_dataset = Dataset.from_dict({"question": test_texts, "label": test_labels})

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/3049 [00:00<?, ? examples/s]

Map:   0%|          | 0/654 [00:00<?, ? examples/s]

Map:   0%|          | 0/654 [00:00<?, ? examples/s]

In [21]:
train_dataset

Dataset({
    features: ['question', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3049
})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base-v2", num_labels=6)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }


# training_args = TrainingArguments(
#     output_dir="./phobert_finetuned",
#     eval_strategy="epoch",
#     save_strategy="no",
#     # save_total_limit=1,
#     logging_steps=10,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=5,
#     weight_decay=0.01,
#     learning_rate=2e-5
#     # load_best_model_at_end=True,
# )
training_args = TrainingArguments(
    output_dir="./phobert_finetuned",
    logging_dir="./logs", 
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    learning_rate=2e-5
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Phuong\.conda\envs\pytorch-gpu\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Phuong\AppData\Local\Temp\ipykernel_5564\3324462921.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.411900,0.304034,0.931193,0.930587
2,0.162900,0.210354,0.949541,0.949449
3,0.052800,0.187801,0.955657,0.955380
4,0.083600,0.176155,0.963303,0.963175
5,0.057100,0.174047,0.961774,0.961674


TrainOutput(global_step=955, training_loss=0.2733618779332226, metrics={'train_runtime': 549.3659, 'train_samples_per_second': 27.75, 'train_steps_per_second': 1.738, 'total_flos': 1002818024040960.0, 'train_loss': 0.2733618779332226, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate(test_dataset)
print(metrics)

{'eval_loss': 0.19434578716754913, 'eval_accuracy': 0.9602446483180428, 'eval_f1': 0.9604083728768114, 'eval_runtime': 4.0432, 'eval_samples_per_second': 161.754, 'eval_steps_per_second': 10.141, 'epoch': 5.0}


In [ ]:
trainer.save_model("./phobert_finetuned")
tokenizer.save_pretrained("./phobert_finetuned")

('./phobert_finetuned\\tokenizer_config.json',
 './phobert_finetuned\\special_tokens_map.json',
 './phobert_finetuned\\vocab.txt',
 './phobert_finetuned\\bpe.codes',
 './phobert_finetuned\\added_tokens.json')